# Assignment-3 Hypothetical Questions

 Set up 

In [1]:
!pip show chromadb

Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: 
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: C:\Users\Palak\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


Ensure the library chromadb is latest. Run the below command

In [3]:
import sys
print(sys.executable)

c:\Users\Palak\Desktop\Assignments\assignment-3\.venv\Scripts\python.exe


In [2]:
import chromadb

from langchain_chroma import Chroma




Below is the api key, endpoint, api version, deployment name for the chat model

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()  

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")



In [7]:
### Before running the command ensure the numpy version is 2.3.4. If not then upgrade numpy as per previous steps in notebook.
### Then do not restart the notebook. If you restart then you will loose all loaded variables. Just click Cancel instad of Restart

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Palak\AppData\Local\Temp\ipykernel_17396\1865801256.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
c:\Users\Palak\Desktop\ass-try\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime
# from google.colab import userdata

 Database creation

In [6]:
pdf_folder_location = "tesla-annual-reports"


In [8]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [22]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [13]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
len(tesla_10k_chunks)

In [9]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [16]:
!pip show sentence-transformers

Name: sentence-transformers
Version: 5.5.1
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: Apache 2.0
Location: C:\Users\palak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, torch, tqdm, transformers, typing_extensions
Required-by: 


In [17]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [10]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [11]:
chromadb_client.heartbeat()

1780551916196978000

In [12]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [13]:
chromadb_client.count_collections()

2

In [32]:
i = 0 # Initialize the starting index for the chunks

while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(5) # Pause for 30 seconds to avoid rate limiting issues with the vector store

In [18]:
chromadb_client

In [14]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [15]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001EAAFE3D650>, search_kwargs={'k': 5})

In [16]:
collection = chromadb_client.get_collection(
    "tesla-10k-2019-to-2023"
)

all_data = collection.get(
    include=["documents", "metadatas"]
)

chunk_ids = all_data["ids"]
chunk_texts = all_data["documents"]

print(len(chunk_ids))
print(chunk_ids[:10])

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


3337
['text_0', 'text_1', 'text_2', 'text_3', 'text_4', 'text_5', 'text_6', 'text_7', 'text_8', 'text_9']


In [28]:
hypothetical_questions_system_message = """
Generate a list of exactly 3 hypothetical questions that the document presented in the input could be used to answer.
Generate only a list of questions, each question in a new line.
Do not number the questions or use bullet points.
Do not mention anything before or after the list.

For EACH chunk provided, generate exactly 3 diverse hypothetical questions that can be answered using that chunk.

Requirements:
- Questions should resemble realistic user queries.
- Each question should focus on a different aspect of the chunk.
- Questions must be answerable from the chunk.
- Do not copy text directly from the chunk.
- Do not skip any chunk.

Output format: In json only

Format:

[
  {
    "chunk_id": "text_1",
    "questions": [
      "question1",
      "question2",
      "question3"
    ]
  }
]




Continue for every chunk provided.

Rules:
- One question per line.
- No numbering or bullet points.
- No explanations or additional text.
"""


In [67]:
user_message_template = """
<Document>
{document}
</Document>
"""

In [68]:
hypothetical_questions = []

In [24]:
batch_collection = Chroma(
    collection_name="tesla-10k-hypothetical-questions-batch",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [18]:
chromadb_client.count_collections()

3

In [25]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023',
 'tesla-10k-hypothetical-questions',
 'tesla-10k-hypothetical-questions-batch']

In [21]:
from openai import OpenAI
import os
client = OpenAI(
    api_key=os.environ["NVIDIA_API_KEY"],
    base_url="https://integrate.api.nvidia.com/v1"
)

In [22]:
source_collection = chromadb_client.get_collection(
    "tesla-10k-2019-to-2023"
)

data = source_collection.get(
    include=["documents", "metadatas"]
)

chunk_ids = data["ids"]
chunk_texts = data["documents"]

print(len(chunk_ids))

3337


In [38]:
import json
import time

from openai import RateLimitError


def generate_batch_questions(batch):

    combined_text = "\n\n".join(
        [
            f"""
CHUNK ID: {item['chunk_id']}

{item['text']}
"""
            for item in batch
        ]
    )

    retries = 10

    for attempt in range(retries):

        try:

            response = client.chat.completions.create(
                model="meta/llama-3.1-70b-instruct",
                response_format={"type": "json_object"},
                messages=[
                    {
                        "role": "system",
                        "content": """
Generate exactly 3 broad hypothetical questions
that could be answered using the information
contained across all provided chunks.

Return ONLY JSON.

{
    "questions":[
        "...",
        "...",
        "..."
    ]
}
"""
                    },
                    {
                        "role": "user",
                        "content": combined_text
                    }
                ],
                temperature=0
            )

            data = json.loads(
                response.choices[0].message.content
            )

            return data["questions"]

        except RateLimitError:

            wait_time = min(
                60 * (attempt + 1),
                600
            )

            print(
                f"Rate limit hit. "
                f"Sleeping {wait_time}s"
            )

            time.sleep(wait_time)

        except Exception as e:

            print(e)

            raise

    raise Exception(
        "Max retries exceeded"
    )

In [39]:
import json


def store_batch_questions(
    batch_id,
    chunk_ids,
    questions
):

    texts = []
    ids = []
    metadatas = []

    for idx, question in enumerate(
        questions,
        start=1
    ):

        texts.append(question)

        ids.append(
            f"{batch_id}_q{idx}"
        )

        metadatas.append(
            {
                "batch_id": batch_id,
                "chunk_count": len(chunk_ids),
                "chunk_ids": json.dumps(
                    chunk_ids
                )
            }
        )

    batch_collection.add_texts(
        texts=texts,
        ids=ids,
        metadatas=metadatas
    )

In [40]:
BATCH_SIZE = 50

total_batches = (
    len(chunk_ids) + BATCH_SIZE - 1
) // BATCH_SIZE

start_batch = load_checkpoint()

print(
    f"Resuming from batch "
    f"{start_batch}"
)

for batch_idx in range(
    start_batch,
    total_batches
):

    start = batch_idx * BATCH_SIZE

    end = min(
        start + BATCH_SIZE,
        len(chunk_ids)
    )

    batch = [
        {
            "chunk_id": cid,
            "text": text
        }
        for cid, text in zip(
            chunk_ids[start:end],
            chunk_texts[start:end]
        )
    ]

    try:

        questions = generate_batch_questions(
            batch
        )

        store_batch_questions(
            batch_id=f"batch_{batch_idx}",
            chunk_ids=[
                x["chunk_id"]
                for x in batch
            ],
            questions=questions
        )

        save_checkpoint(
            batch_idx + 1
        )

        print(
            f"Completed batch "
            f"{batch_idx + 1}/{total_batches}"
        )

    except Exception as e:

        print(
            f"Failed at batch "
            f"{batch_idx}"
        )

        print(e)

        break

Resuming from batch 36
Completed batch 37/67
Completed batch 38/67
Completed batch 39/67
Completed batch 40/67
Completed batch 41/67
Completed batch 42/67
Completed batch 43/67
Completed batch 44/67
Completed batch 45/67
Completed batch 46/67
Completed batch 47/67
Completed batch 48/67
Completed batch 49/67
Completed batch 50/67
Completed batch 51/67
Completed batch 52/67
Completed batch 53/67
Completed batch 54/67
Completed batch 55/67
Completed batch 56/67
Rate limit hit. Sleeping 60s
Completed batch 57/67
Completed batch 58/67
Completed batch 59/67
Completed batch 60/67
Completed batch 61/67
Completed batch 62/67
Completed batch 63/67
Completed batch 64/67
Completed batch 65/67
Completed batch 66/67
Completed batch 67/67


In [41]:
print(batch_collection._collection.count())

201


In [35]:
import json
import os

CHECKPOINT_FILE = "batch_checkpoint.json"


def save_checkpoint(batch_idx):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(
            {"batch_idx": batch_idx},
            f
        )


def load_checkpoint():

    if not os.path.exists(CHECKPOINT_FILE):
        return 0

    with open(CHECKPOINT_FILE, "r") as f:
        return json.load(f)["batch_idx"]

In [36]:
save_checkpoint(36)

In [37]:
print(load_checkpoint())

36


In [43]:
queries = [
    "What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?",
    "How should an analyst investigate the relationship between product defects, warranty or service obligations, customer trust, and brand risk?",
    "What evidence helps determine whether future cash flow depends more on capital expenditure discipline, working capital, or operating income?",
    "Which disclosures help evaluate technology, cybersecurity, data, or AI operational risk even if the user does not explicitly say cybersecurity?"
]

Baseline retrival

In [45]:
for q in queries:

    docs = vectorstore.similarity_search(
        q,
        k=3
    )

    print("=" * 100)
    print(q)

    for i, doc in enumerate(docs):

        print(f"\nChunk {i+1}")
        print(doc.page_content[:500])

What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?

Chunk 1
developing	business	relationships	with	us	if	they	are	not	convinced	that	our	business	will	succeed.	Accordingly,	in	order	to	build	and	maintain	our
business,	we	must	maintain	confidence	among	customers,	suppliers,	analysts,	ratings	agencies	and	other	parties	in	our	long-term	financial	viability	and
business	prospects.	Maintaining	such	confidence	may	be	particularly	complicated	by	certain	factors	including	those	that	are	largely	outside	of	our
control,	such	as	our	limited	operating	history,	custo

Chunk 2
claims	for	vehicle	exposure,	meaning	that	any	product	liability	claims	will	likely	have	to	be	paid	from	company	funds	and	not	by	insurance.
We	will	need	to	maintain	public	credibility	and	confidence	in	our	long-term	business	prospects	in	order	to	succeed.
In	order	to	maintain	and	grow	our	business,	we	must	maintain	credibility	and	confidence	among	cus

In [49]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".

"""

In [50]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [56]:
import json
from groq_client import GroqClient
import os
from dotenv import load_dotenv

load_dotenv()

client = GroqClient()

results = []

for idx, query in enumerate(queries, start=1):

    # Baseline Retrieval
    docs = vectorstore.similarity_search(
        query,
        k=5
    )

    retrieved_contexts = [
        doc.page_content
        for doc in docs
    ]

    context = "\n\n".join(
        retrieved_contexts
    )

    # Generate Answer
    answer = client.generate(
        system_prompt=qna_system_message,
        user_prompt=qna_user_message_template.format(
            context=context,
            question=query
        )
    )

    result = {
        "question_id": idx,
        "original_query": query,
        "retrieved_contexts": retrieved_contexts,
        "generated_answer": answer,
        "retrieval_type": "baseline"
    }

    results.append(result)

    print("=" * 100)
    print(f"Question {idx}")
    print(query)
    print("\nANSWER:")
    print(answer)
    print("=" * 100)

with open(
    "baseline_retrieval_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved baseline_retrieval_results.json"
)

Question 1
What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?

ANSWER:
A board member should ask what risks exist that could hinder Tesla’s ability to meet its production, delivery, and scaling targets, including:

- How limited operating history and customer unfamiliarity with our products might affect demand and execution.  
- Whether there are any anticipated delays in scaling manufacturing capacity, or in delivery and service operations needed to meet demand.  
- How competition and uncertainty about the future of electric vehicles could impact our market share and growth plans.  
- What factors could cause our quarterly production and sales performance to fall short of market expectations.  
- The potential impact of employee attraction, hiring, and retention challenges—especially for key engineering, manufacturing, and leadership talent—on our ability to ramp production.  
- Risks related to supply‑chain

In [57]:
for q in queries:

    docs = batch_collection.similarity_search(
        q,
        k=2
    )

    print("=" * 100)
    print(q)

    for doc in docs:

        print("\nQUESTION:")
        print(doc.page_content)

        print("\nMETADATA:")
        print(doc.metadata)

What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?

QUESTION:
What are the potential risks and challenges associated with Tesla's business operations, including its manufacturing, sales, and energy generation and storage products?

METADATA:
{'batch_id': 'batch_46', 'chunk_count': 50, 'chunk_ids': '["text_2300", "text_2301", "text_2302", "text_2303", "text_2304", "text_2305", "text_2306", "text_2307", "text_2308", "text_2309", "text_2310", "text_2311", "text_2312", "text_2313", "text_2314", "text_2315", "text_2316", "text_2317", "text_2318", "text_2319", "text_2320", "text_2321", "text_2322", "text_2323", "text_2324", "text_2325", "text_2326", "text_2327", "text_2328", "text_2329", "text_2330", "text_2331", "text_2332", "text_2333", "text_2334", "text_2335", "text_2336", "text_2337", "text_2338", "text_2339", "text_2340", "text_2341", "text_2342", "text_2343", "text_2344", "text_2345", "text_2346", "text_2347"

In [64]:
import json

retrieval_results = []

for idx, q in enumerate(queries, start=1):

    # Retrieve top hypothetical batches
    docs = batch_collection.similarity_search(
        q,
        k=1
    )

    chunk_ids = set()

    for doc in docs:

        ids = json.loads(
            doc.metadata["chunk_ids"]
        )

        chunk_ids.update(ids)

    # Retrieve original chunks
    retrieved_docs = source_collection.get(
        ids=list(chunk_ids),
        include=["documents"]
    )

    contexts = retrieved_docs["documents"]

    retrieval_results.append(
        {
            "question_id": idx,
            "original_query": q,
            "retrieved_contexts": contexts,
            "num_contexts": len(contexts)
        }
    )

    print("=" * 100)
    print(f"Question {idx}")
    print(q)
    print(f"\nRetrieved {len(contexts)} chunks")

Question 1
What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?

Retrieved 50 chunks
Question 2
How should an analyst investigate the relationship between product defects, warranty or service obligations, customer trust, and brand risk?

Retrieved 50 chunks
Question 3
What evidence helps determine whether future cash flow depends more on capital expenditure discipline, working capital, or operating income?

Retrieved 50 chunks
Question 4
Which disclosures help evaluate technology, cybersecurity, data, or AI operational risk even if the user does not explicitly say cybersecurity?

Retrieved 50 chunks


In [65]:
retrieval_results
with open(
    "batch_hypothetical_contexts.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        retrieval_results,
        f,
        indent=2,
        ensure_ascii=False
    )

In [75]:
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [85]:
import json

with open(
    "batch_hypothetical_contexts.json",
    "r",
    encoding="utf-8"
) as f:

    retrieval_results = json.load(f)

In [89]:
import os

from langchain_openai import ChatOpenAI

compression_llm = ChatOpenAI(
    api_key=os.environ["NVIDIA_API_KEY"],
    base_url="https://integrate.api.nvidia.com/v1",
    model="meta/llama-3.1-70b-instruct",
    temperature=0,
    max_tokens=2048
)

In [92]:
def compress_context(
    query,
    contexts
):

    full_context = "\n\n".join(
        contexts
    )

    response = compression_llm.invoke(
        f"""
You are a context compression system.

Question:
{query}

Context:
{full_context}

Instructions:
1. Keep only information relevant to answering the question.
2. Remove duplicate information.
3. Preserve facts, numbers, dates, risks, metrics, and disclosures.
4. Produce a concise context.
5. Do not answer the question.

Return only the compressed context.
"""
    )

    return response.content

In [93]:
import json

compressed_results = []

for item in retrieval_results:

    print(
        f"Compressing Question "
        f"{item['question_id']}"
    )

    compressed_context = compress_context(
        query=item["original_query"],
        contexts=item["retrieved_contexts"]
    )

    compressed_results.append(
        {
            "question_id":
                item["question_id"],

            "original_query":
                item["original_query"],

            "compressed_context":
                compressed_context
        }
    )

Compressing Question 1
Compressing Question 2
Compressing Question 3
Compressing Question 4


In [94]:
with open(
    "batch_hypothetical_compressed.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        compressed_results,
        f,
        indent=2,
        ensure_ascii=False
    )

In [95]:
final_results = []

for item in compressed_results:

    answer = client.generate(
        system_prompt=qna_system_message,
        user_prompt=qna_user_message_template.format(
            context=item["compressed_context"],
            question=item["original_query"]
        )
    )

    result = {
        "question_id": item["question_id"],
        "original_query": item["original_query"],
        "compressed_context": item["compressed_context"],
        "generated_answer": answer
    }

    final_results.append(result)

    print("=" * 100)
    print(f"Question {item['question_id']}")
    print(item["original_query"])
    print("\nANSWER:\n")
    print(answer)
    print("=" * 100)

Question 1
What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?

ANSWER:

A board member should probe each of the identified risk areas, for example:

- How are we managing supply‑chain and logistics challenges, and what contingency plans are in place for component constraints?
- What steps are being taken to prevent or quickly recover from factory shutdowns?
- How are we addressing labor shortages at our production sites?
- What is the impact of COVID‑19 (or any other health epidemic) on our operations, and how are we mitigating it?
- Are our warranty reserves and insurance coverage sufficient to cover potential liabilities?
- Do we have any debt‑covenant restrictions that could limit our ability to scale, and how are we monitoring them?
- How are currency‑exchange fluctuations being hedged or otherwise managed?
- What is the status of any intellectual‑property infringement claims and their potential impact?
- 

In [96]:
import json

with open(
    "batch_hypothetical_final_answers.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved batch_hypothetical_final_answers.json"
)

Saved batch_hypothetical_final_answers.json
